# Fine-tune Qwen3-8B on your real JARVIS conversations

This notebook fine-tunes **Qwen3-8B** (the same model family JARVIS uses via Ollama, `qwen3:8b`) on the **real user/assistant turns** JARVIS collected while you talked to it.

### Before you start
1. Talk to JARVIS for a while so `data/conversations.jsonl` grows.
2. Prepare the training file locally:
   ```
   python tools/prepare_dataset.py
   ```
3. Upload the produced `data/dataset.json` to this Colab runtime (Files pane / sidebar).
4. Runtime > Change runtime type > **T4 GPU**.
5. Runtime > Run all.

At the end you get a GGUF file + a Modelfile to install into Ollama on your PC.

## 1. Install Unsloth

In [ ]:
%%capture
import os, re
if "COLAB_" not in "".join(os.environ.keys()):
    !pip install unsloth
else:
    import torch
    v = re.match(r'[\d]{1,}\.[\d]{1,}', str(torch.__version__)).group(0)
    xformers = 'xformers==' + {'2.10':'0.0.34','2.9':'0.0.33.post1','2.8':'0.0.32.post2'}.get(v, "0.0.34")
    !pip install sentencepiece protobuf "datasets==4.3.0" "huggingface_hub>=0.34.0" hf_transfer
    !pip install --no-deps unsloth_zoo bitsandbytes accelerate {xformers} peft trl triton unsloth
    !pip install --no-deps --upgrade "torchao>=0.16.0"
!pip install transformers==4.56.2
!pip install --no-deps trl==0.22.2

## 2. Load Qwen3-8B (4-bit)

In [ ]:
from unsloth import FastLanguageModel
import torch

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = "unsloth/Qwen3-8B-unsloth-bnb-4bit",
    max_seq_length = 2048,   # Short answers — JARVIS speaks 1-2 sentences.
    load_in_4bit = True,     # Fits the free T4 GPU.
    full_finetuning = False,
)

## 3. Add LoRA adapters

In [ ]:
model = FastLanguageModel.get_peft_model(
    model,
    r = 16,
    target_modules = ["q_proj", "k_proj", "v_proj", "o_proj",
                      "gate_proj", "up_proj", "down_proj"],
    lora_alpha = 16,
    lora_dropout = 0,
    bias = "none",
    use_gradient_checkpointing = "unsloth",
    random_state = 3407,
)

## 4. Load & format your real conversation dataset

Uses the `dataset.json` that `tools/prepare_dataset.py` produced. Format is ShareGPT (human/gpt roles + optional system), which `standardize_sharegpt` understands natively.

In [ ]:
from datasets import load_dataset
from unsloth.chat_templates import standardize_sharegpt

raw = load_dataset("json", data_files="dataset.json", split="train")
print("Examples loaded:", len(raw))
print(raw[0])

In [ ]:
# Normalize to the HF conversation format Unsloth expects.
ds = standardize_sharegpt(raw)

# JARVIS speaks with thinking disabled (config sends "think": False), so we
# format every example in NON-thinking mode to match runtime behaviour.
def format_convo(examples):
    texts = []
    systems = examples.get("system", [None] * len(examples["conversations"]))
    for convos, system in zip(examples["conversations"], systems):
        messages = []
        if system:
            messages.append({"role": "system", "content": system})
        messages.extend(convos)
        texts.append(
            tokenizer.apply_chat_template(
                messages,
                tokenize=False,
                enable_thinking=False,
            )
        )
    return {"text": texts}

ds = ds.map(format_convo, batched=True).select_columns(["text"])

# 90/10 train/eval split from your real data.
split = ds.train_test_split(test_size=0.1, seed=3407)
train_ds, eval_ds = split["train"], split["test"]
print("Train examples:", len(train_ds))
print("Eval examples:", len(eval_ds))
print("\nFirst formatted example:\n", train_ds[0]["text"][:400])

## 5. Train

In [ ]:
# @markdown Steps to run. Your dataset is small (real conversations), so even
# @markdown 100-300 steps adapts the style well. Increase if you logged a lot.
max_steps = 120  # @param {type:"integer"}

In [ ]:
from trl import SFTTrainer, SFTConfig

trainer = SFTTrainer(
    model = model,
    tokenizer = tokenizer,
    train_dataset = train_ds,
    eval_dataset = eval_ds,
    args = SFTConfig(
        dataset_text_field = "text",
        per_device_train_batch_size = 2,
        gradient_accumulation_steps = 4,
        warmup_steps = 5,
        max_steps = max_steps,
        learning_rate = 2e-4,
        logging_steps = 1,
        optim = "adamw_8bit",
        weight_decay = 0.001,
        lr_scheduler_type = "linear",
        seed = 3407,
        report_to = "none",
        output_dir = "outputs",
        evaluation_strategy = "steps",
        eval_steps = max(1, max_steps // 5),
        save_strategy = "no",
    ),
)
trainer.train()

## 6. Test it before exporting

In [ ]:
FastLanguageModel.for_inference(model)
messages = [
    {"role": "system", "content": "You are JARVIS, a concise and helpful desktop voice assistant. Answer in one to two short sentences, spoken aloud. Give direct answers; avoid long explanations unless the user explicitly asks for detail."},
    {"role": "user", "content": "What can you help me with today?"},
]
inputs = tokenizer.apply_chat_template(
    messages, tokenize=True, add_generation_prompt=True,
    enable_thinking=False, return_tensors="pt",
).to("cuda")
outputs = model.generate(input_ids=inputs, max_new_tokens=64, temperature=0.7)
print(tokenizer.decode(outputs[0], skip_special_tokens=True))

## 7. Export to GGUF + build the Ollama Modelfile

In [ ]:
# Merge LoRA into the base and quantize to q4_k_m (good quality / small size
# for Ollama on a normal PC).
model.save_pretrained_gguf("jarvis_finetune", tokenizer, quantization_method="q4_k_m")

In [ ]:
import glob

gguf = glob.glob("jarvis_finetune/*.gguf")[0]
print("GGUF file:", gguf)
print("Size (GB):", round(os.path.getsize(gguf) / 1e9, 2))

In [ ]:
# @markdown Name for the Ollama model, e.g. `qwen3:8b-jarvis`.
model_name = "qwen3:8b-jarvis"  # @param {type:"string"}

modelfile = f'''FROM {os.path.basename(gguf)}
SYSTEM """You are JARVIS, a concise and helpful desktop voice assistant. Answer in one to two short sentences, spoken aloud. Give direct answers; avoid long explanations unless the user explicitly asks for detail."""
PARAMETER temperature 0.7
PARAMETER top_p 0.8
PARAMETER top_k 20
PARAMETER num_ctx 4096
'''

with open("Modelfile", "w", encoding="utf-8") as fh:
    fh.write(modelfile)
print("Modelfile written.")

In [ ]:
# Download BOTH files to your PC (Files pane > right-click > Download):
#   1. {gguf}     <- the fine-tuned model
#   2. Modelfile  <- Ollama recipe

# On your PC, in the folder containing both files:
print("# On your PC:")
print(f"    ollama create {model_name} -f Modelfile")
print(f"    ollama run {model_name}")
print()
print("# Then tell JARVIS to use it — edit config.py:")
print(f"    OLLAMA_MODEL = \"{model_name}\"")